In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

np.random.seed(42)

In [2]:
BASE_DIR = Path(
    "paper_implementation"
)

PROFILE_DIR = Path(
    "daily_profiles_24h"
)

MODE4_DIR = (
    BASE_DIR /
    "theft_simulation" /
    "mode_4"
)

MODE4_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
selected_df = pd.read_csv(
    BASE_DIR /
    "selected_150_meters" /
    "selected_150_meter_ids.csv"
)

fraud_df = pd.read_csv(
    BASE_DIR /
    "area_assignments" /
    "fraud_consumers.csv"
)

selected_meters = (
    selected_df["Meter"]
    .astype(str)
    .tolist()
)

fraud_meters = set(
    fraud_df["Meter"]
    .astype(str)
    .tolist()
)

print(
    "Selected Consumers:",
    len(selected_meters)
)

print(
    "Fraud Consumers:",
    len(fraud_meters)
)

Selected Consumers: 150
Fraud Consumers: 27


In [4]:
hour_cols = [
    f"HOUR_{i}"
    for i in range(24)
]

In [6]:
meter_intervals = {}

for meter in fraud_meters:

    t1 = np.random.randint(
        0,
        22
    )

    t2 = np.random.randint(
        t1 + 2,
        24
    )

    meter_intervals[meter] = (
        t1,
        t2
    )

print(
    list(
        meter_intervals.items()
    )[:5]
)

[('6270', (20, 23)), ('47281', (19, 23)), ('40768', (6, 19)), ('41675', (7, 23)), ('42633', (2, 17))]


In [7]:
total_modified = 0

log_records = []

for meter in selected_meters:

    df = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    if meter in fraud_meters:

        t1, t2 = meter_intervals[meter]

        for row_idx in df.index:

            date = df.loc[
                row_idx,
                "DATE"
            ]

            for hour in hour_cols:

                hour_num = int(
                    hour.split("_")[1]
                )

                x_it = float(
                    df.loc[
                        row_idx,
                        hour
                    ]
                )

                # Equation (11)

                if t1 < hour_num < t2:

                    x_prime = 0

                else:

                    x_prime = x_it

                log_records.append({

                    "Meter":
                    meter,

                    "Date":
                    date,

                    "Hour":
                    hour,

                    "Actual_Reading":
                    x_it,

                    "t1":
                    t1,

                    "t2":
                    t2,

                    "Final_Output":
                    x_prime
                })

                df.loc[
                    row_idx,
                    hour
                ] = x_prime

                total_modified += 1

    df.to_csv(
        MODE4_DIR /
        f"{meter}.csv",
        index=False
    )

log_df = pd.DataFrame(
    log_records
)

log_df.to_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_4_generation_log.csv",
    index=False
)

print(
    "Modified readings:",
    total_modified
)

print(
    "Log rows:",
    len(log_df)
)

Modified readings: 20088
Log rows: 20088


In [8]:
print(
    "Files Generated:",
    len(
        list(
            MODE4_DIR.glob("*.csv")
        )
    )
)

log_df = pd.read_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_4_generation_log.csv"
)

print(
    "Log Rows:",
    len(log_df)
)

Files Generated: 150
Log Rows: 20088


In [9]:
normal_meters = [
    m for m in selected_meters
    if m not in fraud_meters
]

unchanged_count = 0

for meter in normal_meters:

    original = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    mode4 = pd.read_csv(
        MODE4_DIR /
        f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode4[hour_cols].values
    )

    if same:
        unchanged_count += 1

print(
    "Unchanged Normal Consumers:",
    unchanged_count,
    "/",
    len(normal_meters)
)

Unchanged Normal Consumers: 123 / 123


In [10]:
modified_count = 0

for meter in fraud_meters:

    original = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    mode4 = pd.read_csv(
        MODE4_DIR /
        f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode4[hour_cols].values
    )

    if not same:
        modified_count += 1

print(
    "Modified Fraud Consumers:",
    modified_count,
    "/",
    len(fraud_meters)
)

Modified Fraud Consumers: 27 / 27


In [11]:
check = []

for _, row in log_df.iterrows():

    hour_num = int(
        row["Hour"].split("_")[1]
    )

    if row["t1"] < hour_num < row["t2"]:

        check.append(
            row["Final_Output"] == 0
        )

    else:

        check.append(
            row["Final_Output"]
            ==
            row["Actual_Reading"]
        )

all(check)

True

In [14]:
log_df.head(24)

,Meter,Date,Hour,Actual_Reading,t1,t2,Final_Output
0,6270,2018-07-01,HOUR_0,1.2354,20,23,1.2354
1,6270,2018-07-01,HOUR_1,1.1880,20,23,1.1880
2,6270,2018-07-01,HOUR_2,0.8430,20,23,0.8430
3,6270,2018-07-01,HOUR_3,0.5580,20,23,0.5580
4,6270,2018-07-01,HOUR_4,0.5826,20,23,0.5826
5,6270,2018-07-01,HOUR_5,0.5688,20,23,0.5688
6,6270,2018-07-01,HOUR_6,0.5652,20,23,0.5652
7,6270,2018-07-01,HOUR_7,0.5376,20,23,0.5376
8,6270,2018-07-01,HOUR_8,0.8574,20,23,0.8574
9,6270,2018-07-01,HOUR_9,0.6018,20,23,0.6018
